# 05 — Monthly precipitationsheds for one city, one year

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/blob/main/notebooks/05_yearly_moisture_metrics.ipynb)

**What we're investigating.** A 12-panel monthly view of where one
city's rain came from across a single year, taking Santiago in 2019
as the example. Each panel shows the moisture-source field for that
calendar month, ranked as a **spatial CDF** so small absolute
differences across seasons still read clearly: 0% = the dominant
source pixel, 100% = the long tail. Read across the grid to see the
source field migrate with the seasons.

**Dataset.** AguaTrack v1, monthly aggregate store at
`AguaTrackSA/AguaTrack-ARCO-SA-Aggregated`. We pick a single year
(`STUDY_YEAR = 2019`) and a single city so the read is tiny — under
5 MB into RAM — and Colab-friendly.

**How to cite.** See the [repo README](https://github.com/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial#how-to-cite).

## Step 1 — Configuration

Everything you might want to edit lives in this single cell:

- **HuggingFace dataset** — `AguaTrackSA/AguaTrack-ARCO-SA-Aggregated`
  (both the monthly and yearly stores).
- **Study year** — defaults to 2019. Any year in 1990–2019 works.
- **Target city** — Santiago (Chile) by default. Set
  `TARGET_LAT` / `TARGET_LON` to any other location to rerun the
  12-panel monthly CDF there.
- **Zoom** — set `ZOOM = True` for a continental-scale view restricted
  to ±10° around the target city.

In [ ]:
HF_REVISION = "main"

STUDY_YEAR = 2019

TARGET_NAME = "Viedma"
TARGET_LAT = -40.82
TARGET_LON = -63.00

ZOOM = False

AGUATRACK_MONTHLY_URL = (
    "hf://datasets/AguaTrackSA/AguaTrack-ARCO-SA-Aggregated"
    "/AguaTrack_ARCO_SA_monthly.zarr"
)
AGUATRACK_YEARLY_URL = (
    "hf://datasets/AguaTrackSA/AguaTrack-ARCO-SA-Aggregated"
    "/AguaTrack_ARCO_SA_yearly.zarr"
)

## Step 2 — Install dependencies (Colab only)

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        'install -q cartopy cmcrameri "xarray>=2026" "zarr>=3" '
        "fsspec huggingface_hub dask",
    )

## Step 3 — Imports and plotting style

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import xarray as xr

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
})

OUT = Path("outputs/yearly_metrics")
OUT.mkdir(parents=True, exist_ok=True)

## Step 4 — Find the tag cell nearest to the target city

`tag_lat`/`tag_lon` are static metadata of the tracking domain. A
Euclidean argmin in (lat, lon) at 0.25° is precise enough — no need
for a great-circle distance.

In [ ]:
ds_monthly_full = xr.open_zarr(AGUATRACK_MONTHLY_URL,
                               storage_options={"revision": HF_REVISION})

dist_sq = (ds_monthly_full.tag_lat - TARGET_LAT) ** 2 \
        + (ds_monthly_full.tag_lon - TARGET_LON) ** 2
tag_idx = int(dist_sq.argmin())
tag_lon = float(ds_monthly_full.tag_lon.isel(tagging_mask=tag_idx))
tag_lat = float(ds_monthly_full.tag_lat.isel(tagging_mask=tag_idx))
print(
    f"  target=({TARGET_LAT}, {TARGET_LON})  "
    f"nearest tag=({tag_lat:.2f}, {tag_lon:.2f})  idx={tag_idx}"
)

## Step 5 — Pull the 12 monthly source maps for the study year

Pangeo idiom: select the single tag and the 12 months of `STUDY_YEAR`
*before* triggering any read, then materialise the (12, lat, lon)
array in one go.

In [ ]:
from dask.diagnostics import ProgressBar

monthly_one_tag = (
    ds_monthly_full.isel(tagging_mask=tag_idx)
    .sel(time=str(STUDY_YEAR))
)

with ProgressBar():
    e_track_monthly = monthly_one_tag.e_track.load()  # (time=12, lat, lon)

print(f"loaded {e_track_monthly.sizes['time']} monthly source maps "
      f"({e_track_monthly.nbytes / 1e6:.1f} MB in RAM)")

## Step 6 — Spatial CDF helper

Each panel is ranked independently. CDF(pixel) = "this pixel and all
stronger-contributing pixels together account for X% of the panel
total." 0% = a single dominant source pixel; 100% = the long tail.
When you draw a contour at, say, 30%, you get the boundary of "the
pixels that supply 30% of the source".

In [ ]:
def spatial_cdf(da: xr.DataArray) -> xr.DataArray:
    """Convert a 2-D positive-valued source map into a per-panel rank CDF in %."""
    vals_flat = da.values.flatten()
    valid_mask = ~np.isnan(vals_flat) & (vals_flat > 0)
    valid_vals = vals_flat[valid_mask]
    sort_idx = np.argsort(valid_vals)[::-1]
    cumsum_pct = np.cumsum(valid_vals[sort_idx]) / np.sum(valid_vals[sort_idx]) * 100
    cdf_flat = np.full_like(vals_flat, np.nan, dtype=float)
    cdf_flat[np.where(valid_mask)[0][sort_idx]] = cumsum_pct
    return xr.DataArray(cdf_flat.reshape(da.values.shape), coords=da.coords, dims=da.dims)

## Step 7 — The 12-panel monthly precipitationshed grid

Two rows of six panels, one panel per calendar month. Each panel is a
rank-CDF map (0% = strongest source pixel, 100% = long tail), with the
target city marked by a red dot. The 0–10% contour traces the core
moisture-source region for that month.

In [ ]:
levels = np.arange(0, 101, 10)


def style_monthly_ax(ax, left_labels=True, bottom_labels=True):
    """Decorate one panel of the 2x6 monthly grid."""
    ax.coastlines(resolution="50m", color="black", linewidth=0.8)
    if ZOOM:
        ax.set_xlim(tag_lon - 10, tag_lon + 10)
        ax.set_ylim(tag_lat - 10, tag_lat + 10)
    else:
        ax.set_xlim(float(ds_monthly_full.longitude.min()),
                    float(ds_monthly_full.longitude.max()))
        ax.set_ylim(float(ds_monthly_full.latitude.min()),
                    float(ds_monthly_full.latitude.max()))
    gl = ax.gridlines(draw_labels=True, linewidth=0.0)
    gl.top_labels = gl.right_labels = False
    gl.left_labels = left_labels
    gl.bottom_labels = bottom_labels
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 10))
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, 10))


fig, axes = plt.subplots(
    2, 6, figsize=(20, 8),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
axes = axes.flatten()

cf = None
for i in range(12):
    ax = axes[i]
    da_month = e_track_monthly.isel(time=i)
    cdf_2d = spatial_cdf(da_month)
    cf = cdf_2d.plot.contourf(
        ax=ax, levels=levels, cmap=cmc.batlowW,
        transform=ccrs.PlateCarree(), add_colorbar=False, extend="max",
    )
    month_str = np.datetime_as_string(da_month.time.values, unit="M")
    ax.set_title(month_str)
    ax.scatter(tag_lon, tag_lat, color="red", s=15,
               transform=ccrs.PlateCarree(), zorder=5)
    # Lat labels only on the leftmost column (i % 6 == 0); lon labels on
    # the bottom row (i >= 6).
    style_monthly_ax(ax, left_labels=(i % 6 == 0), bottom_labels=(i >= 6))

# One shared horizontal colour bar at the bottom.
fig.subplots_adjust(left=0.04, right=0.98, top=0.90, bottom=0.10,
                    hspace=0.10, wspace=0.04)
cbar_ax = fig.add_axes([0.15, 0.03, 0.7, 0.018])
fig.colorbar(cf, cax=cbar_ax, orientation="horizontal",
             ticks=levels, extend="both",
             label="Cumulative Moisture Contribution (%)")
fig.suptitle(
    f"Monthly Precipitationsheds — {TARGET_NAME} "
    f"({tag_lon:.2f}, {tag_lat:.2f}) — {STUDY_YEAR}",
    fontsize=20, y=0.97,
)

fig.savefig(OUT / f"fig1_monthly_precipitationsheds_{STUDY_YEAR}.png",
            bbox_inches="tight", dpi=150)
plt.show()

ds_monthly_full.close()